切块有个绕不开的根本矛盾：
块切小了，召回准，但喂给 LLM 的内容不够；块切大了，内容够，但召回被稀释、不准。
小块语义聚焦，一算相似度就命中，可就那么两句话，模型答不全。大块信息完整，可里面掺了太多别的内容，embedding 一平均，语义就糊了，该召回的时候反而排不上去。
怎么破？父子分块，也叫 Small-to-Big。

思路很妙：召回用小块，喂给模型用大块。
子块（小）
只拿来做 embedding 和召回。短、语义聚焦，召回准。
父块（大）
一旦某个子块被召回命中，实际喂给 LLM 的，是这个子块所属的父块。上下文完整。

LangChain 的 ParentDocumentRetriever 把这套逻辑封好了——子块进向量库（Milvus），父块进 docstore：

In [6]:
from langchain_core.documents import Document
docs = [
    Document(page_content="""X3 智能手环售后服务说明：
自客户签收商品当日起算，无理由退换货期限为15天。
15天内，商品外观无损伤、配件齐全，可申请全额退换货。
超过15天不再支持无理由退换。
整机免费保修时长为12个月。
电池属于消耗配件，保修期6个月。
人为进水、摔落、私自拆机造成的损坏不在保修范围。
售后客服工作时间：周一‑周五 9:00‑18:00。"""),

    Document(page_content="""X5 智能手表售后政策：
X5无理由退换货时效为7天。
签收7天以内，未激活、无划痕，支持7天无理由。
激活之后不再支持无理由退换。
整机保修期18个月。
屏幕碎裂属于人为损坏，不在免费保修范围内。
如需寄回维修，运费前两次由我方承担。"""),

    Document(page_content="""Y1蓝牙耳机使用注意事项：
Y1蓝牙耳机续航单次6小时，充电仓总续航30小时。
充电仓充满大约需要2小时。
退换货政策：签收30天内无理由退换。
耳机丢失单只不在保修补发范围。""")
]

In [11]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1500)   # 父块，喂 LLM
child_splitter  = RecursiveCharacterTextSplitter(chunk_size=300)    # 子块，做召回

embedding= HuggingFaceEmbeddings(
    model_name=r"G:\力扣代码集\langchain实现RAG\download_model\m3e-base",
)

vector_store = Chroma(
    embedding_function=embedding,
    persist_directory="./chroma_db"
)

retriever = ParentDocumentRetriever(
    vectorstore=vector_store,      # 子块的向量存这里（bge-m3 embedding）
    docstore=InMemoryStore(),      # 父块原文存这里
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)
retriever.add_documents(docs)

# 召回时：用 300 字的子块匹配（准），返回 1500 字的父块（够）喂给 DeepSeek
results = retriever.invoke("Y1的退换货政策是几天？")
print("召回结果：", results[0].page_content)

召回结果： Y1蓝牙耳机使用注意事项：
Y1蓝牙耳机续航单次6小时，充电仓总续航30小时。
充电仓充满大约需要2小时。
退换货政策：签收30天内无理由退换。
耳机丢失单只不在保修补发范围。


表格和代码块，作为独立父块整体处理——召回到就整块给出去，绝不切开。这就是前面那条铁律的落地方式。

还有一件容易被忽略的小事：metadata 挂载。给每个块挂上来源文件、标题路径、页码、类型。它至少有两个用处：一是答完能溯源，告诉用户"这条来自手册第 12 页售后章节"；二是召回时能做元数据过滤，比如只在"售后"章节里检索。

把全流程端到端串起来，就是这么一条链路：

原始 PDF
  → unstructured 解析 + 表格转 Markdown
  → clean_text 清洗页眉页脚/软换行
  → MarkdownHeaderTextSplitter 切出带标题路径的父块
  → RecursiveCharacterTextSplitter 切出子块
  → bge-m3 embed 子块
  → Milvus 存子块向量 + docstore 存父块原文
  → 召回子块 → 映射回父块 → 拼上 metadata → 喂 DeepSeek 生成

PART 06：一份可复用的"解析→切块"检查清单
讲了这么多，压成一张上手就能用的清单。下次接到一份文档，照着走：

1. 先分类，别急着上代码。

有没有文本层？（PyMuPDF 取出来是空的，就是扫描件，得走 OCR）
有没有表格？（有就必须单独处理）
是不是多栏？（是就重点检查阅读顺序）
2. 按类型选解析武器。

电子版 PDF：PyMuPDF 取正文 + pdfplumber 抽表格
扫描件：OCR + 版面分析（PP-Structure）
懒得手写、类型杂：unstructured 或 MinerU 一体化
3. 表格一律转 Markdown/HTML，单独成块，绝不切开。

4. 清洗，但别过度。

去页眉页脚（按重复率检测）、合并软换行
清洗规则跟着文档类型走，代码/列表手下留情

5. 切块看文档结构。

结构化文档优先 header split，每块带标题路径 metadata
散文用递归切块，实在需要再上语义切块
每个块都挂上来源、页码、类型
6. 召回准但答不全，就上父子分块。

7. 上线前，务必抽检。

抽 20 个真实问题，人工核对召回块里的表格、关键数字对不对
别只看召回分数——分数漂亮不代表内容对，这正是这篇从头到尾在说的事
这七步走完，你的文本才算真正"配得上"后面那套花哨的检索。

结尾：胜负手在最不起眼的地方
我们太容易把注意力放在光鲜的环节——更强的模型、更花哨的检索、更复杂的 Agent。可 RAG 真正的胜负手，往往藏在一张没人愿意多看一眼的表格里，藏在"文档怎么变成文本"这最不起眼的一步里。

RAG 的准确率，不是在检索时挣来的，是在文档进库那一刻就定死的。

你喂给模型的每一份脏数据，都会在某个你看不见的地方，变成一个理直气壮的错误答案。

互动时间：你踩过最坑的一次文档解析是什么？是 PDF 表格错乱，是扫描件 OCR 串行，还是那些永远对不齐的多栏排版？评论区聊聊，我挑几个典型的下次专门拆